In [0]:
claims_df = spark.table("main.healthcare_project_pipe.silver_claims")

members_df = spark.table("main.healthcare_project_pipe.silver_members")

providers_df = spark.table("main.healthcare_project_pipe.silver_providers")

pharmacy_df = spark.table("main.healthcare_project_pipe.silver_pharmacy")

In [0]:
gold_df = claims_df.join(
    members_df,
    "member_id",
    "left"
)

In [0]:
from pyspark.sql import functions as F

monthly_summary = gold_df.groupBy(
    "claim_year",
    "claim_month",
    "state"
).agg(
    F.sum("claim_amount").alias("total_claim_amount"),
    F.countDistinct("member_id").alias("total_members"),
    F.countDistinct("claim_id").alias("total_claims")
)
display(monthly_summary)


In [0]:
monthly_summary.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("main.healthcare_project_pipe.monthly_claim_summary")